# Simulation 7: reproduce in Colab (ChroniCare-AI, protocol v0.6)

**What this notebook does:** downloads the Simulation 7 code from the public repository, **re-runs it from scratch** with the same seeds, and compares the results with the published `sim7_*.csv` files.

- Everything is **simulated**: no patient data are used.
- It takes about **30 to 45 minutes** (in the environment where it was first run it took about 30 minutes with 2 cores). Do not close the tab.
- The last cell states whether the result **reproduces or not**. If there are differences, do not fix them: copy the output as it is and it will be reported in the protocol.

Order: run the cells from top to bottom (Environment, Run, Analyze, Compare).

In [1]:
# 1. Environment: download the code and print library versions
import os, sys, platform, subprocess
if not os.path.exists('/content/repo'):
    !git clone -q --depth 1 https://github.com/alygallolopez-ship-it/Seminario-doctoral-r6s10714-Metodos-de-Investigacion-cientifica /content/repo
%cd /content/repo/simulation_full_pipeline
import numpy, scipy, pandas, statsmodels, joblib
print('python', platform.python_version(), '| cores:', os.cpu_count())
for m in (numpy, scipy, pandas, statsmodels, joblib):
    print(m.__name__, m.__version__)
print()
print(open('requirements_sim6.txt').read())
print('Commit of the code that will be run:')
!git -C /content/repo log -1 --format='%h %ad %s' --date=short

/content/repo/simulation_full_pipeline
python 3.13.18 | cores: 2
numpy 2.4.6
scipy 1.16.3
pandas 2.2.3
statsmodels 0.15.0
joblib 1.6.0

numpy==2.4.6
pandas==2.2.3
statsmodels==0.15.0
scipy==1.16.3
joblib==1.6.0
matplotlib==3.10.0

Commit of the code that will be run:
6904c12 2026-11-18 README: documenta un duplicado residual en la hoja de Scopus


In [2]:
# 2. Run the simulation (the long part). Writes R.pkl and RC.pkl to /content/sim7_out
!mkdir -p /content/sim7_out
%time !python run_sim7.py /content/sim7_out

techo listo 15.7 min
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:  8.8min
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed: 63.9min
[Parallel(n_jobs=2)]: Done 288 out of 288 | elapsed: 151.0min finished
malla lista 166.7 min
CPU times: user 20.9 s, sys: 2.31 s, total: 23.2 s
Wall time: 2h 46min 48s


In [3]:
# 3. Analyze: produce the CSV files from the replicates (same calculations as the published ones)
!python analizar7.py /content/sim7_out /content/sim7_out

VALORES CRITICOS
   n resultado  vc_a05          celda_a05  vc_a025         celda_a025
 500         R  0.1145 ruido 0.15, c 0.36   0.1184  ruido 0.6, c 0.36
 500         F  0.0120  ruido 0.15, c 0.0   0.0155  ruido 0.15, c 0.0
 500        Rb  0.1082 ruido 0.15, c 0.36   0.1130 ruido 0.15, c 0.36
 738         R  0.1109 ruido 0.15, c 0.36   0.1132 ruido 0.15, c 0.36
 738         F  0.0078  ruido 0.15, c 0.0   0.0109  ruido 0.15, c 0.0
 738        Rb  0.1026 ruido 0.15, c 0.36   0.1064 ruido 0.15, c 0.36
1000         R  0.1085 ruido 0.15, c 0.36   0.1115 ruido 0.15, c 0.36
1000         F  0.0055  ruido 0.15, c 0.0   0.0086  ruido 0.15, c 0.0
1000        Rb  0.1000 ruido 0.15, c 0.36   0.1030 ruido 0.15, c 0.36
2000         R  0.1042 ruido 0.15, c 0.36   0.1060 ruido 0.15, c 0.36
2000         F  0.0012  ruido 0.15, c 0.0   0.0033  ruido 0.15, c 0.0
2000        Rb  0.0939 ruido 0.15, c 0.36   0.0961 ruido 0.15, c 0.36
3000         R  0.1023 ruido 0.15, c 0.36   0.1042 ruido 0.15, c 0.36
300

In [4]:
# 4. Compare with the files published in the repository
import glob, hashlib, numpy as np, pandas as pd
rows = []
for f in sorted(glob.glob('/content/repo/simulation_full_pipeline/sim7_*.csv')):
    name = os.path.basename(f); g = '/content/sim7_out/' + name
    if not os.path.exists(g):
        rows.append((name, 'NOT GENERATED', None)); continue
    a, b = pd.read_csv(f), pd.read_csv(g)
    if a.shape != b.shape or list(a.columns) != list(b.columns):
        rows.append((name, 'DIFFERENT SHAPE', None)); continue
    num = a.select_dtypes('number').columns
    dif = float(np.nanmax(np.abs(a[num].values - b[num].values))) if len(num) else 0.0
    txt = all((a[c].astype(str) == b[c].astype(str)).all() for c in a.columns if c not in num)
    rows.append((name, 'IDENTICAL' if (dif < 1e-9 and txt) else ('NEARLY IDENTICAL (diff <= 1e-6)' if (dif < 1e-6 and txt) else 'DIFFERENT'), dif))
res = pd.DataFrame(rows, columns=['file', 'result', 'max_numeric_difference'])
print(res.to_string(index=False))
print()
print('SHA-256 fingerprints of the files you just generated:')
for g in sorted(glob.glob('/content/sim7_out/sim7_*.csv')):
    print(hashlib.sha256(open(g, 'rb').read()).hexdigest()[:16], os.path.basename(g))
ok = all(r[1] in ('IDENTICAL', 'NEARLY IDENTICAL (diff <= 1e-6)') for r in rows) and len(rows) > 0
print()
print('REPRODUCED' if ok else 'NOT REPRODUCED: copy this full output and send it without changes')

                        file    result  max_numeric_difference
             sim7_Dmin_F.csv IDENTICAL            1.006140e-16
  sim7_poder_suficiencia.csv IDENTICAL            1.110223e-16
sim7_sensibilidad7_regla.csv IDENTICAL            1.110223e-16
             sim7_tamano.csv IDENTICAL            0.000000e+00
        sim7_techo_dbeta.csv DIFFERENT            1.333333e-02
   sim7_valores_criticos.csv IDENTICAL            9.887924e-17

SHA-256 fingerprints of the files you just generated:
88ad142c116cd1d2 sim7_Dmin_F.csv
a7f161defbd95a10 sim7_poder_suficiencia.csv
ad3845c26e8535fe sim7_sensibilidad7_regla.csv
ba8b0298c46eead9 sim7_tamano.csv
ecc47327cc1fe16c sim7_techo_dbeta.csv
fe9055f32020030b sim7_valores_criticos.csv

NOT REPRODUCED: copy this full output and send it without changes


In [5]:
# 5. Key figures cited in the protocol (for a visual check)
vc = pd.read_csv('/content/sim7_out/sim7_valores_criticos.csv')
print(vc[['n', 'resultado', 'vc_a025']].round(3).to_string(index=False))
tam = pd.read_csv('/content/sim7_out/sim7_tamano.csv'); print(); print(tam.head(8).round(3).to_string(index=False))

   n resultado  vc_a025
 500         R    0.118
 500         F    0.015
 500        Rb    0.113
 738         R    0.113
 738         F    0.011
 738        Rb    0.106
1000         R    0.112
1000         F    0.009
1000        Rb    0.103
2000         R    0.106
2000         F    0.003
2000        Rb    0.096
3000         R    0.104
3000         F    0.001
3000        Rb    0.094
5000         R    0.102
5000         F   -0.001
5000        Rb    0.091

  n  ruido    c resultado  tamano  error_MC
500   0.15 0.00         R   0.000     0.000
500   0.15 0.00         F   0.025     0.005
500   0.15 0.00        Rb   0.000     0.000
500   0.15 0.12         R   0.000     0.000
500   0.15 0.12         F   0.012     0.003
500   0.15 0.12        Rb   0.000     0.000
500   0.15 0.24         R   0.000     0.000
500   0.15 0.24         F   0.001     0.001


**When it finishes:** take a screenshot of the output of cell 4 (or copy the text) and send it to Claude. The protocol's AI-use declaration will then state that Simulation 7 was reproduced by the researcher in Google Colab.